### Simulating Recurrent Neural Network (RNN) State Updates.

In [ ]:
# Dummy RNN model: takes an input and a state, and returns a new state
def model(input_value, prev_state):
    # For simplicity, just combine the input and previous state
    return input_value + prev_state

# Example input sequence (e.g., token Ids)
inputs = [1, 2, 3, 4, 5]

# Initial state (can be zeros in real models)
state = 0

# Simulate RNN processing each input step-by-step
for i, input_value in enumerate(inputs):
    # Call the model with the current input and previous state
    state = model(input_value, state)
    print(f"Step {i}: input = {input_value}, state = {state}")

###  Basic encoder implementation

In [7]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_size: int, embedding_dim: int, hidden_dim: int):
        """
        input_size: vocabulary size (e.g., 10000)
        embedding_dim: size of each embedding vector (e.g., 256)
        hidden_dim: size of hidden layer in GRU (Gated Recurrent Unit) (e.g., 512)
        """
        super(Encoder, self).__init__()

        # Define encoder embedding layer
        self.embedding = nn.Embedding(input_size, embedding_dim)

        # Define the GRU (Gated Recurrent Unit) layer
        self.rnn = nn.GRU(embedding_dim, hidden_dim)

    def forward(self, token_ids):
        # Get the embeddings from input token IDs
        embeddings = self.embedding(token_ids)

        # Get the output and hidden state (context vector) from the GRU
        _, state = self.rnn(embeddings)

        return state  # Context vector is the hidden state of the last layer

### Basic decoder implementation

In [8]:
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(self, output_size: int, embedding_dim: int, hidden_dim: int):
        """
        output_size: size of the output vocabulary
        embedding_dim: size of the embedding vectors
        hidden_dim: size of the hidden state in the GRU
        """
        super(Decoder, self).__init__()

        self.embedding = nn.Embedding(output_size, embedding_dim)
        self.rnn = nn.GRU(embedding_dim + hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_size)

    def forward(self, input_token, state):
        # Embed the input token
        embedded = self.embedding(input_token)

        # Pass through the GRU
        output, state = self.rnn(embedded, state)

        # Generate final output token probabilities
        output = self.fc_out(output)

        return output, state

### Using encoder and decoder

In [ ]:
# SOS - start of sentence, EOS - end of sentence
input_vocabulary = ["<SOS>", "<EOS>", "Enjoy", "the", "show"]  # English sentence - tokens
output_vocabulary = ["<SOS>", "<EOS>", "Disfruta", "del", "espectáculo"]  # Spanish translation

input_size = len(input_vocabulary)   # 5
output_size = len(output_vocabulary) # 5

# Initialize encoder and decoder
encoder = Encoder(input_size, embedding_dim=3, hidden_dim=3)
decoder = Decoder(output_size, embedding_dim=3, hidden_dim=3)

# Tokenized input: <SOS> Enjoy the show <EOS>
token_ids = [0, 2, 3, 4, 1]

# Encode the input sequence
context_vector = encoder(token_ids)  # output: [0.27, -0.95, 0.15]

# <SOS> token is used to start decoding the output sequence
sos_token_id = 0

# Get the first output token
logits, context_vector = decoder(sos_token_id, hidden=context_vector)
# logits = [0.73, 0.23, 0.15, 0.13, 0.28], context_vector = [0.38, -0.92, 0.03]
first_output_token_id = logits.argmax(1)  # output: 0

# Get the second output token
logits, context_vector = decoder(first_output_token_id, context_vector)
# logits = [0.12, 0.29, 0.81, 0.05, 0.21], context_vector = [-0.26, -0.81, 0.17]
second_output_token_id = logits.argmax(1)  # output: 2 (Disfruta)

# Get the third output token
logits, context_vector = decoder(second_output_token_id, context_vector)
# logits = [0.01, 0.31, 0.25, 0.79, 0.11], context_vector = [0.17, 0.77, -0.52]
third_output_token_id = logits.argmax(1)  # output: 3 (del)

# Final output sequence (token IDs)
output = [0, 2, 3, 4, 1]  # <SOS> Disfruta del espectáculo <EOS>